# Genre-Specific Screenplay LoRA Training
**Model:** `meta-llama/Llama-3.2-3B-Instruct`  
**Method:** QLoRA via Unsloth  
**Hardware:** Kaggle 2x T4 (16GB total)  
**Genres:** Drama → Horror → Sci-Fi → Comedy (sequential)

Each genre trains a **separate LoRA adapter** pushed to HF Hub:
```
SatyaMoulika/llama-3.2-drama-lora
SatyaMoulika/llama-3.2-horror-lora
SatyaMoulika/llama-3.2-scifi-lora
SatyaMoulika/llama-3.2-comedy-lora
```

## 1 · Install Dependencies

In [ ]:
%%capture
import os
if 'KAGGLE_KERNEL_RUN_TYPE' in os.environ:
    # Kaggle environment — install from pip
    !pip install unsloth datasets huggingface_hub tqdm
else:
    # Colab fallback
    !pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
    !pip install datasets huggingface_hub tqdm

## 2 · HuggingFace Login

In [ ]:
from huggingface_hub import login, whoami
login()   # paste your HF write token
print('Logged in as:', whoami()['name'])

## 3 · Config

In [ ]:
HF_USERNAME  = 'SatyaMoulika'
BASE_MODEL   = 'meta-llama/Llama-3.2-3B-Instruct'

# Dataset repos (one per genre, created in Phase 1)
GENRE_DATASETS = {
    'drama':  'SatyaMoulika/imsdb-drama-screenplay',
    'horror': 'SatyaMoulika/imsdb-horror-screenplay',
    'scifi':  'SatyaMoulika/imsdb-scifi-screenplay',
    'comedy': 'SatyaMoulika/imsdb-comedy-screenplay',
}

# LoRA hyperparams
LORA_R           = 16
LORA_ALPHA       = 32
LORA_DROPOUT     = 0.05
TARGET_MODULES   = ['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj']

# Training hyperparams
MAX_SEQ_LENGTH   = 1024
BATCH_SIZE       = 2       # per device; effective = 2 x 2 GPUs = 4
GRAD_ACCUM       = 4       # effective batch = 16
EPOCHS           = 3
LR               = 2e-4
WARMUP_RATIO     = 0.05
LR_SCHEDULER     = 'cosine'
SAVE_STEPS       = 50
LOGGING_STEPS    = 10
MAX_STEPS        = -1      # -1 = full epochs; set e.g. 200 to cap for testing

OUTPUT_DIR       = '/kaggle/working/checkpoints'

## 4 · GPU Check

In [ ]:
import torch

print(f'CUDA available : {torch.cuda.is_available()}')
print(f'GPU count      : {torch.cuda.device_count()}')
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f'  GPU {i}: {props.name}  {props.total_memory / 1e9:.1f} GB')

## 5 · Load Base Model + Tokenizer (once, reused across genres)

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name      = BASE_MODEL,
    max_seq_length  = MAX_SEQ_LENGTH,
    dtype           = None,    # auto: float16 on T4
    load_in_4bit    = True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print('Base model loaded.')
print(f'Tokenizer pad token: {tokenizer.pad_token}')

## 6 · Prompt Formatter

In [ ]:
EOS = tokenizer.eos_token

def format_prompt(sample: dict) -> dict:
    """
    Converts a dataset row into the full training string.
    The `text` column from Phase 1 already has the right format
    (### Instruction / ### Screenplay) — just append EOS.
    """
    sample['text'] = sample['text'] + EOS
    return sample

## 7 · Training Function (one genre)

In [ ]:
import os
import gc
import torch
from datasets import load_dataset
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments

def train_genre(genre: str, dataset_repo: str, base_model, base_tokenizer):
    print(f'\n{"="*55}')
    print(f'  Training: {genre.upper()}')
    print(f'  Dataset : {dataset_repo}')
    print(f'{"="*55}')

    # ── 1. Load dataset ──────────────────────────────────────
    ds = load_dataset(dataset_repo)
    train_ds = ds['train'].map(format_prompt)
    val_ds   = ds['validation'].map(format_prompt)
    print(f'  Train: {len(train_ds)} samples   Val: {len(val_ds)} samples')

    # ── 2. Attach fresh LoRA adapter ─────────────────────────
    # Each genre gets its own adapter; base weights are shared & frozen
    lora_model = FastLanguageModel.get_peft_model(
        base_model,
        r                   = LORA_R,
        lora_alpha          = LORA_ALPHA,
        lora_dropout        = LORA_DROPOUT,
        target_modules      = TARGET_MODULES,
        bias                = 'none',
        use_gradient_checkpointing = 'unsloth',  # saves ~30% VRAM
        random_state        = 42,
    )

    # ── 3. Training args ─────────────────────────────────────
    genre_output = os.path.join(OUTPUT_DIR, genre)
    os.makedirs(genre_output, exist_ok=True)

    args = TrainingArguments(
        output_dir                  = genre_output,
        num_train_epochs            = EPOCHS,
        max_steps                   = MAX_STEPS,
        per_device_train_batch_size = BATCH_SIZE,
        gradient_accumulation_steps = GRAD_ACCUM,
        learning_rate               = LR,
        lr_scheduler_type           = LR_SCHEDULER,
        warmup_ratio                = WARMUP_RATIO,
        fp16                        = not torch.cuda.is_bf16_supported(),
        bf16                        = torch.cuda.is_bf16_supported(),
        logging_steps               = LOGGING_STEPS,
        save_steps                  = SAVE_STEPS,
        save_total_limit            = 2,
        evaluation_strategy         = 'steps',
        eval_steps                  = SAVE_STEPS,
        load_best_model_at_end      = True,
        metric_for_best_model       = 'eval_loss',
        report_to                   = 'none',   # set 'wandb' if you use W&B
        optim                       = 'adamw_8bit',
        weight_decay                = 0.01,
        seed                        = 42,
    )

    # ── 4. Trainer ───────────────────────────────────────────
    trainer = SFTTrainer(
        model           = lora_model,
        tokenizer       = base_tokenizer,
        train_dataset   = train_ds,
        eval_dataset    = val_ds,
        dataset_text_field = 'text',
        max_seq_length  = MAX_SEQ_LENGTH,
        args            = args,
    )

    # ── 5. Train ─────────────────────────────────────────────
    trainer.train()

    # ── 6. Push adapter to HF Hub ────────────────────────────
    adapter_repo = f'{HF_USERNAME}/llama-3.2-{genre}-lora'
    lora_model.push_to_hub(adapter_repo, tokenizer=base_tokenizer)
    print(f'  Adapter pushed → https://huggingface.co/{adapter_repo}')

    # ── 7. Log training loss curve ───────────────────────────
    log_history = trainer.state.log_history
    train_losses = [(e['step'], e['loss'])      for e in log_history if 'loss'      in e]
    eval_losses  = [(e['step'], e['eval_loss']) for e in log_history if 'eval_loss' in e]

    # ── 8. Free adapter from memory before next genre ────────
    del lora_model, trainer
    gc.collect()
    torch.cuda.empty_cache()

    return {'train_losses': train_losses, 'eval_losses': eval_losses}

## 8 · Train All Genres Sequentially

In [ ]:
all_logs = {}

for genre, dataset_repo in GENRE_DATASETS.items():
    logs = train_genre(genre, dataset_repo, model, tokenizer)
    all_logs[genre] = logs
    print(f'\n{genre.upper()} training complete.')

print('\nAll genres trained.')

## 9 · Plot Training & Eval Loss per Genre

In [ ]:
import matplotlib.pyplot as plt

GENRE_COLORS = {
    'drama':  '#4C9BE8',
    'horror': '#E84C4C',
    'scifi':  '#9B4CE8',
    'comedy': '#F4A645',
}

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
axes = axes.flatten()

for ax, (genre, logs) in zip(axes, all_logs.items()):
    color = GENRE_COLORS[genre]

    if logs['train_losses']:
        steps, losses = zip(*logs['train_losses'])
        ax.plot(steps, losses, label='Train loss', color=color, linewidth=1.5)

    if logs['eval_losses']:
        steps, losses = zip(*logs['eval_losses'])
        ax.plot(steps, losses, label='Val loss', color=color,
                linewidth=1.5, linestyle='--')

    ax.set_title(f'{genre.capitalize()} — Loss Curve')
    ax.set_xlabel('Step')
    ax.set_ylabel('Loss')
    ax.legend()
    ax.grid(alpha=0.3)

plt.suptitle('QLoRA Training — Loss per Genre', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('/kaggle/working/loss_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Loss curves saved.')

## 10 · Sanity Check — Sample Generation per Adapter

In [ ]:
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

TEST_PROMPTS = {
    'drama':  'Scene heading: INT. HOSPITAL ROOM - NIGHT\nStory context: A father visits his estranged daughter for the last time.',
    'horror': 'Scene heading: EXT. ABANDONED FARMHOUSE - MIDNIGHT\nStory context: A group of friends realise they are not alone.',
    'scifi':  'Scene heading: INT. SPACESHIP BRIDGE - DAY\nStory context: The crew discovers an alien signal from a dying star.',
    'comedy': 'Scene heading: INT. OFFICE BREAK ROOM - MORNING\nStory context: Two coworkers fight over the last cup of coffee.',
}

def generate_scene(adapter_repo: str, prompt_body: str, genre: str) -> str:
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
    base = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL, quantization_config=bnb, device_map='auto', torch_dtype=torch.float16)
    tok  = AutoTokenizer.from_pretrained(BASE_MODEL)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token

    ft = PeftModel.from_pretrained(base, adapter_repo)
    ft.eval()

    prompt = (
        f'### Instruction:\nWrite a {genre} screenplay scene.\n'
        f'{prompt_body}\n\n### Screenplay:\n'
    )
    inputs = tok(prompt, return_tensors='pt').to(ft.device)
    with torch.no_grad():
        out = ft.generate(**inputs, max_new_tokens=200, do_sample=True,
                          temperature=0.8, top_p=0.9,
                          pad_token_id=tok.eos_token_id)
    full = tok.decode(out[0], skip_special_tokens=True)
    scene = full.split('### Screenplay:')[-1].strip()

    del base, ft
    import gc; gc.collect(); torch.cuda.empty_cache()
    return scene


for genre, prompt_body in TEST_PROMPTS.items():
    adapter_repo = f'{HF_USERNAME}/llama-3.2-{genre}-lora'
    print(f'\n{"-"*55}')
    print(f'GENRE: {genre.upper()}')
    print(f'PROMPT: {prompt_body[:80]}...')
    scene = generate_scene(adapter_repo, prompt_body, genre)
    print(f'OUTPUT:\n{scene[:600]}')

## 11 · Training Summary

In [ ]:
print('='*55)
print('TRAINING SUMMARY')
print('='*55)
for genre in GENRE_DATASETS:
    adapter_repo = f'{HF_USERNAME}/llama-3.2-{genre}-lora'
    logs = all_logs[genre]
    final_train = logs['train_losses'][-1][1]  if logs['train_losses'] else 'N/A'
    final_eval  = logs['eval_losses'][-1][1]   if logs['eval_losses']  else 'N/A'
    print(f"""
{genre.upper()}
  Final train loss : {final_train}
  Final eval loss  : {final_eval}
  Adapter repo     : https://huggingface.co/{adapter_repo}
""")